# Annotation status

Class balance of the expert annotations, to decide which classes need more work.
Works offline from the BIIGLE *CSV image annotation reports*. No API requests.

**To refresh:** put the report CSVs (or the zips) in `reports/csv/` - one file per volume - then *Run All*.
Figures are saved to `outputs/<date>/`.

## 1. Settings

In [ ]:
from pathlib import Path
import datetime as dt

BASE = Path.cwd()
CSV_DIR = BASE / "reports" / "csv"     # one BIIGLE report per volume: <volume id>-<name>.csv (zips are unpacked)
OUT_DIR = BASE / "outputs"
EXCLUDED_USERS = {"Ali Loulidi", "Nils Piechaud"}   # their annotations are ignored

# Readiness tiers (annotations per class)
TIERS = [
    (1000, "Ready (1000+)", "#0ca30c"),
    (300, "Usable (300-999)", "#fab219"),
    (100, "Needs more (100-299)", "#ec835a"),
    (0, "Too few (<100)", "#d03b3b"),
]

RUN_DATE = dt.date.today().isoformat()
RUN_DIR = OUT_DIR / RUN_DATE
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the reports

In [ ]:
import re, zipfile
import pandas as pd

for z in CSV_DIR.glob("*.zip"):                      # unpack any zip dropped into the folder
    with zipfile.ZipFile(z) as f:
        for m in f.infolist():
            if m.filename.lower().endswith(".csv"):
                (CSV_DIR / Path(m.filename).name).write_bytes(f.read(m))
    z.unlink()

COLS = ["label_id", "label_name", "label_hierarchy", "firstname", "lastname",
        "image_id", "filename", "shape_name", "annotation_id", "created_at"]

def short_name(slug):
    m = re.match(r"(r\d+vl\d+)", slug)
    return m.group(1).upper() if m else "-".join(slug.split("-")[:4])

frames = []
for f in sorted(CSV_DIR.glob("*.csv")):
    vid, slug = f.stem.split("-", 1)
    d = pd.read_csv(f, usecols=COLS, dtype={"firstname": str, "lastname": str})
    d["volume_id"], d["volume"] = int(vid), short_name(slug)
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)

raw["user"] = (raw["firstname"].fillna("") + " " + raw["lastname"].fillna("")).str.strip()
df = raw[~raw["user"].isin(EXCLUDED_USERS)].copy()
df["class"] = df["label_name"].str.replace("\xa0", " ").str.split().str.join(" ")
df["hierarchy"] = df["label_hierarchy"].fillna(df["label_name"]).str.replace("\xa0", " ")
df["group"] = df["hierarchy"].str.split(" > ").str[0].str.strip()
df["created_at"] = pd.to_datetime(df["created_at"])

print(f"{len(frames)} volume reports read")
print(f"{len(df):,} expert annotations | {df['class'].nunique()} classes | "
      f"{df['volume_id'].nunique()} volumes | {df['image_id'].nunique():,} images")
print("annotators:", ", ".join(f"{u} ({n:,})" for u, n in df["user"].value_counts().items()))

## 3. Counts

In [ ]:
def tier(n):
    return next(t for t in TIERS if n >= t[0])

vol_order = df.groupby("volume")["volume_id"].first().sort_values().index.tolist()

cls = df["class"].value_counts().rename_axis("Class").reset_index(name="Annotations")
cls["Status"] = [tier(n)[1] for n in cls["Annotations"]]

matrix = pd.crosstab(df["class"], df["volume"]).reindex(columns=vol_order).loc[cls["Class"]]
users = df["user"].value_counts().index.tolist()

print(cls["Status"].value_counts().reindex([t[1] for t in TIERS], fill_value=0).to_string())
display(cls.head(30))

## 4. Figures

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from matplotlib.patches import Patch

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
def new_fig(w, h):
    fig, ax = plt.subplots(figsize=(w, h), dpi=120, facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#b9b8b2")
    ax.tick_params(axis="y", length=0, colors=INK)
    ax.tick_params(axis="x", colors=INK2, labelsize=8)
    ax.grid(axis="x", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    return fig, ax

def save(fig, name):
    path = RUN_DIR / f"{name}_{RUN_DATE}.png"
    fig.savefig(path, facecolor=SURFACE, bbox_inches="tight")
    plt.show()

# annotations per class
n = len(cls)
fig, ax = new_fig(11, 0.17 * n + 1.6)
vals = cls["Annotations"]
ax.barh(range(n), vals, color=[tier(v)[2] for v in vals], height=0.78)
ax.set_xscale("log")
ax.set_xlim(0.8, vals.max() * 2.5)
ax.set_ylim(n - 0.4, -0.6)
ax.set_yticks(range(n))
ax.set_yticklabels([s if len(s) < 55 else s[:52] + "..." for s in cls["Class"]], fontsize=6.6)
for i, v in enumerate(vals):
    ax.text(v * 1.08, i, f"{v:,}", va="center", fontsize=6, color=INK2)
ax.tick_params(axis="x", which="both", labeltop=True, top=True)
ax.set_xlabel("Annotations (log scale)", color=INK2, fontsize=8)
tc = cls["Status"].value_counts()
ax.legend(handles=[Patch(color=c, label=f"{l}: {tc.get(l, 0)} classes") for _, l, c in TIERS],
          loc="lower right", fontsize=8, frameon=False)
ax.set_title(f"Annotations per class  ({vals.sum():,} annotations, {n} classes, {RUN_DATE})",
             loc="left", fontsize=10, color=INK, pad=24)
save(fig, "annotations_per_class")

# annotations per volume, by annotator
pv = pd.crosstab(df["volume"], df["user"])[users]
pv = pv.loc[pv.sum(axis=1).sort_values().index]
fig, ax = new_fig(10, 0.42 * len(pv) + 1.4)
left = pd.Series(0, index=pv.index)
for i, u in enumerate(users):
    ax.barh(pv.index, pv[u], left=left, color=SERIES[i % 8], height=0.7, edgecolor=SURFACE, linewidth=2, label=u)
    left += pv[u]
for i, v in enumerate(left):
    ax.text(v + left.max() * 0.01, i, f"{v:,}", va="center", fontsize=8, color=INK2)
ax.set_xlim(0, left.max() * 1.12)
ax.legend(loc="lower right", fontsize=8, frameon=False)
ax.set_title("Annotations per volume", loc="left", fontsize=10, color=INK)
save(fig, "annotations_per_volume")

# top 40 classes x volume
top = matrix.head(40)
fig, ax = plt.subplots(figsize=(0.9 * top.shape[1] + 5, 0.26 * len(top) + 1.8), dpi=120, facecolor=SURFACE)
cmap = LinearSegmentedColormap.from_list("blues", BLUES)
cmap.set_bad("#f0efec")
ax.imshow(top.where(top > 0).values, aspect="auto", cmap=cmap, norm=LogNorm(vmin=1, vmax=max(top.values.max(), 2)))
ax.set_xticks(range(top.shape[1]))
ax.set_xticklabels(top.columns, rotation=40, ha="right", fontsize=8, color=INK)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([s if len(s) < 50 else s[:47] + "..." for s in top.index], fontsize=7.5, color=INK)
for (i, j), v in pd.DataFrame(top.values).stack().items():
    if v:
        ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=6.5, color="#ffffff" if v >= 300 else INK)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
ax.set_title("Top 40 classes per volume (grey = none)", loc="left", fontsize=10, color=INK)
save(fig, "classes_per_volume")
print("figures saved to", RUN_DIR)